In [1]:
!pip install datasets transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-c

In [2]:
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Load the dataset
ds = load_dataset("edaschau/bitcoin_news")
df = ds['train'].to_pandas()

# Data filtering
df['date'] = pd.to_datetime(df['time_unix'], unit="s")
df = df.sort_values(by='date', ascending=True)
df.set_index('date', inplace=True)

df = df[['title', 'article_text', 'url']]

print(df.head(3))
print(df.tail(3))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/695 [00:00<?, ?B/s]

BTC_yahoo.csv:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80806 [00:00<?, ? examples/s]

                                                                 title  \
date                                                                     
2011-06-22 10:56:00  Compromised account leads to massive Bitcoin s...   
2012-02-01 18:02:32          Bitcoin May Be The Currency Of The Future   
2012-03-22 19:23:56  Should Africa Adopt a Shared Currency? And Sho...   

                                                          article_text  \
date                                                                     
2011-06-22 10:56:00  Bitcoin, for those not aware, is a completely ...   
2012-02-01 18:02:32  Have you heard of Bitcoin? If you're a fan of ...   
2012-03-22 19:23:56  Dekstop /Flickr I wrote on Monday about Sweden...   

                                                                   url  
date                                                                    
2011-06-22 10:56:00  https://finance.yahoo.com/news/2011-06-22-comp...  
2012-02-01 18:02:32  https://finance.ya

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ElKulako/cryptobert")
model = AutoModelForSequenceClassification.from_pretrained("ElKulako/cryptobert")

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/932 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [4]:
# Ensure model and device are set correctly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

Using device: cuda


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [5]:
def analyze_sentiment_in_batches(texts, batch_size):
    sentiment_labels = []
    sentiment_scores = []
    sentiment_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch_texts = texts[i:i + batch_size]
        if not batch_texts:  # Ensure non-empty batch
            continue

        # Tokenize and move inputs to GPU
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {key: val.to(device) for key, val in inputs.items()}

        # Perform inference on GPU
        with torch.no_grad():
            outputs = model(**inputs)

        # Compute softmax for probability scores
        probs = F.softmax(outputs.logits, dim=-1).cpu()  # Move to CPU before conversion

        # Get predicted class labels
        predicted_classes = torch.argmax(probs, dim=1).cpu()  # Move to CPU before indexing
        batch_labels = [model.config.id2label[idx.item()] for idx in predicted_classes]

        # Get confidence scores (probability of the predicted class)
        batch_scores = probs.gather(1, predicted_classes.unsqueeze(1)).squeeze(1).tolist()

        # Store probabilities of all classes
        batch_probs = probs.tolist()

        sentiment_labels.extend(batch_labels)
        sentiment_scores.extend(batch_scores)
        sentiment_probs.extend(batch_probs)

    return sentiment_labels, sentiment_scores, sentiment_probs

In [6]:
# Run sentiment analysis
df['sentiment_labels'], df['sentiment_confidence'], df['sentiment_probabilities'] = analyze_sentiment_in_batches(
    df['article_text'].tolist(), batch_size=256
)

# Convert probability lists into structured columns for each sentiment class
num_classes = len(model.config.id2label)
sentiment_columns = [f"sentiment_prob_class_{i}" for i in range(num_classes)]
df[sentiment_columns] = pd.DataFrame(df['sentiment_probabilities'].tolist(), index=df.index)

# Drop probability column if not needed
df.drop(columns=['sentiment_probabilities'], inplace=True)

# Display first few rows for verification
print(df.head())
df.head(3)

Processing Batches:   0%|          | 0/316 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Processing Batches: 100%|██████████| 316/316 [45:36<00:00,  8.66s/it]


                                                                 title  \
date                                                                     
2011-06-22 10:56:00  Compromised account leads to massive Bitcoin s...   
2012-02-01 18:02:32          Bitcoin May Be The Currency Of The Future   
2012-03-22 19:23:56  Should Africa Adopt a Shared Currency? And Sho...   
2012-08-22 23:58:00  MasterCard denies BitCoin card rumors, BitInst...   
2012-09-07 13:38:01    Claim of Romney taxes theft a puzzling whodunit   

                                                          article_text  \
date                                                                     
2011-06-22 10:56:00  Bitcoin, for those not aware, is a completely ...   
2012-02-01 18:02:32  Have you heard of Bitcoin? If you're a fan of ...   
2012-03-22 19:23:56  Dekstop /Flickr I wrote on Monday about Sweden...   
2012-08-22 23:58:00  MasterCard shoots down BitCoin debit card rumo...   
2012-09-07 13:38:01  WASHINGTON (AP) 

,title,article_text,url,sentiment_labels,sentiment_confidence,sentiment_prob_class_0,sentiment_prob_class_1,sentiment_prob_class_2
date,,,,,,,,
2011-06-22 10:56:00,Compromised account leads to massive Bitcoin s...,"Bitcoin, for those not aware, is a completely ...",https://finance.yahoo.com/news/2011-06-22-comp...,Neutral,0.592729,0.001572,0.592729,0.405698
2012-02-01 18:02:32,Bitcoin May Be The Currency Of The Future,Have you heard of Bitcoin? If you're a fan of ...,https://finance.yahoo.com/news/bitcoin-may-cur...,Neutral,0.548736,0.000427,0.548736,0.450838
2012-03-22 19:23:56,Should Africa Adopt a Shared Currency? And Sho...,Dekstop /Flickr I wrote on Monday about Sweden...,https://finance.yahoo.com/news/africa-adopt-sh...,Neutral,0.593259,0.003214,0.593259,0.403527


In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the dataset to a CSV file in Google Drive
df.to_csv('/content/drive/My Drive/Colab Notebooks/bitcoin_news_data.csv')

Mounted at /content/drive
